## Neural Networks

This notebook demonstrates how to build, train, and evaluate neural networks for both regression and classification tasks using TensorFlow and Keras.

1. Setup
2. Feed-Forward Neural Networks (Regression)
3. Feed-Forward Neural Networks (Classification)
4. Recurrent Neural Networks
5. Convolutional Neural Networks


### 1. Setup and Imports

First, let's import all the necessary libraries.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, classification_report
import seaborn as sns

### 2. Regression Example

#### 2.1. Generate Synthetic Data

We'll create a simple dataset where the output `y` is a non-linear function of input `x`, with some added noise.

In [ ]:
# Generate synthetic data for regression
np.random.seed(42)
X_reg = np.random.rand(500, 1) * 10
y_reg = 2 * X_reg**2 - 5 * X_reg + 3 + np.random.randn(500, 1) * 5

plt.figure(figsize=(8, 6))
plt.scatter(X_reg, y_reg, alpha=0.6)
plt.title('Synthetic Regression Data')
plt.xlabel('X')
plt.ylabel('y')
plt.grid(True)
plt.show()

#### 2.2. Preprocess Data

Scaling the features is crucial for neural networks.

In [ ]:
# Split data into training and testing sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Scale features
scaler_X_reg = StandardScaler()
X_train_reg_scaled = scaler_X_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_X_reg.transform(X_test_reg)

scaler_y_reg = StandardScaler()
y_train_reg_scaled = scaler_y_reg.fit_transform(y_train_reg)
y_test_reg_scaled = scaler_y_reg.transform(y_test_reg)

#### 2.3. Build the Regression Model

We'll create a simple feed-forward neural network.

In [ ]:
# Build the model
model_reg = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_reg_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1) # Output layer for regression has one neuron with no activation (linear activation)
])

# Compile the model
model_reg.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_reg.summary()

#### 2.4. Train the Regression Model

Training the model involves feeding it the training data and allowing it to learn the patterns.

In [ ]:
# Train the model
history_reg = model_reg.fit(X_train_reg_scaled, y_train_reg_scaled, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_reg.history['loss'], label='Training Loss')
plt.plot(history_reg.history['val_loss'], label='Validation Loss')
plt.title('Regression Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_reg.history['mae'], label='Training MAE')
plt.plot(history_reg.history['val_mae'], label='Validation MAE')
plt.title('Regression Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.show()

#### 2.5. Evaluate the Regression Model

We'll evaluate the model on the test set and visualize its predictions.

In [ ]:
# Evaluate the model on the test set
loss_reg, mae_reg = model_reg.evaluate(X_test_reg_scaled, y_test_reg_scaled, verbose=0)
print(f'Test Loss (MSE): {loss_reg:.4f}')
print(f'Test MAE: {mae_reg:.4f}')

# Make predictions
y_pred_reg_scaled = model_reg.predict(X_test_reg_scaled)
y_pred_reg = scaler_y_reg.inverse_transform(y_pred_reg_scaled)

# Calculate R-squared and MSE for unscaled predictions
mse_unscaled_reg = mean_squared_error(y_test_reg, y_pred_reg)
r2_unscaled_reg = r2_score(y_test_reg, y_pred_reg)
print(f'Test MSE (unscaled): {mse_unscaled_reg:.4f}')
print(f'Test R^2 (unscaled): {r2_unscaled_reg:.4f}')

# Visualize predictions vs actual values
plt.figure(figsize=(10, 7))
plt.scatter(X_test_reg, y_test_reg, label='Actual Values', alpha=0.7)
plt.scatter(X_test_reg, y_pred_reg, label='Predicted Values', alpha=0.7, marker='x')
plt.title('Regression Model: Actual vs Predicted Values')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

# Plot predictions sorted by X
sorted_indices = np.argsort(X_test_reg.flatten())
plt.figure(figsize=(10, 7))
plt.plot(X_test_reg[sorted_indices], y_test_reg[sorted_indices], label='Actual Values')
plt.plot(X_test_reg[sorted_indices], y_pred_reg[sorted_indices], label='Predicted Values', linestyle='--')
plt.title('Regression Model: Actual vs Predicted Values (Sorted by X)')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

### 3. Classification Example

#### 3.1. Generate Synthetic Data

We'll create a synthetic dataset for binary classification (two classes).

In [ ]:
# Generate synthetic data for classification (e.g., two half-moons)
from sklearn.datasets import make_moons
X_clf, y_clf = make_moons(n_samples=500, noise=0.15, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap='viridis', alpha=0.7)
plt.title('Synthetic Classification Data (Two Moons)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()

#### 3.2. Preprocess Data

Similar to regression, scaling is important.

In [ ]:
# Split data into training and testing sets
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Scale features
scaler_X_clf = StandardScaler()
X_train_clf_scaled = scaler_X_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_X_clf.transform(X_test_clf)

#### 3.3. Build the Classification Model

For binary classification, the output layer will have one neuron with a `sigmoid` activation.

In [ ]:
# Build the model
model_clf = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_clf_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Output layer for binary classification with sigmoid activation
])

# Compile the model
model_clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_clf.summary()

#### 3.4. Train the Classification Model

We'll train the classification model.

In [ ]:
# Train the model
history_clf = model_clf.fit(X_train_clf_scaled, y_train_clf, epochs=100, batch_size=32, validation_split=0.2, verbose=0)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_clf.history['loss'], label='Training Loss')
plt.plot(history_clf.history['val_loss'], label='Validation Loss')
plt.title('Classification Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Crossentropy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_clf.history['accuracy'], label='Training Accuracy')
plt.plot(history_clf.history['val_accuracy'], label='Validation Accuracy')
plt.title('Classification Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

#### 3.5. Evaluate the Classification Model

We'll evaluate the model's accuracy, confusion matrix, and classification report.

In [ ]:
# Evaluate the model on the test set
loss_clf, accuracy_clf = model_clf.evaluate(X_test_clf_scaled, y_test_clf, verbose=0)
print(f'Test Loss (Binary Crossentropy): {loss_clf:.4f}')
print(f'Test Accuracy: {accuracy_clf:.4f}')

# Make predictions
y_pred_proba_clf = model_clf.predict(X_test_clf_scaled)
y_pred_clf = (y_pred_proba_clf > 0.5).astype(int)

# Confusion Matrix
cm = confusion_matrix(y_test_clf, y_pred_clf)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test_clf, y_pred_clf))

#### 3.6. Visualize Decision Boundary

To better understand the classification model, let's visualize its decision boundary.

In [ ]:
# Function to plot decision boundary
def plot_decision_boundary(X, y, model, scaler):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    # Scale the grid points
    grid_points_scaled = scaler.transform(np.c_[xx.ravel(), yy.ravel()])

    # Predict probabilities for each grid point
    Z = model.predict(grid_points_scaled, verbose=0).reshape(xx.shape)
    Z = (Z > 0.5).astype(int)

    plt.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, s=20, edgecolor='k', cmap='viridis')
    plt.title('Classification Decision Boundary')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

plt.figure(figsize=(10, 7))
plot_decision_boundary(X_clf, y_clf, model_clf, scaler_X_clf)

### 4. Recurrent Neural Network (RNN) Example (Time Series Prediction)

We'll use a simple RNN to predict the next value in a synthetic time series.

#### 4.1. Generate Synthetic Sequence Data

We'll create a sine wave with some noise as our time series data.

In [ ]:
# Generate synthetic time series data
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))  #   wave 1
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20)) # + wave 2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)   # + noise
    return series[..., np.newaxis]

n_steps = 50
series = generate_time_series(10000, n_steps + 1) # Generate data for 10000 sequences, each of length n_steps + 1
X_rnn = series[:, :n_steps]
y_rnn = series[:, -1]

# Visualize a few sequences
plt.figure(figsize=(10, 6))
plt.plot(X_rnn[0].flatten(), label='Input Sequence (first 50 steps)')
plt.plot(n_steps, y_rnn[0], 'ro', label='Target (last step)')
plt.title('Synthetic Time Series Example')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

#### 4.2. Preprocess Data

Splitting and reshaping for RNN input (samples, timesteps, features).

In [ ]:
# Split data into training and testing sets
X_train_rnn, X_test_rnn, y_train_rnn, y_test_rnn = train_test_split(X_rnn, y_rnn, test_size=0.2, random_state=42)

# RNNs typically handle sequences directly, but ensure correct shape (samples, timesteps, features)
# Our data is already in this shape from generation: (batch_size, n_steps, 1)

#### 4.3. Build the RNN Model

We'll use a simple `SimpleRNN` layer for sequence prediction.

In [ ]:
# Build the model
model_rnn = keras.Sequential([
    layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]), # Input shape [None, 1] for variable sequence length, 1 feature
    layers.SimpleRNN(20),
    layers.Dense(1) # Output layer for predicting a single value
])

# Compile the model
model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_rnn.summary()

#### 4.4. Train the RNN Model

In [ ]:
# Train the model
history_rnn = model_rnn.fit(X_train_rnn, y_train_rnn, epochs=20, batch_size=32, validation_split=0.2, verbose=0)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_rnn.history['loss'], label='Training Loss')
plt.plot(history_rnn.history['val_loss'], label='Validation Loss')
plt.title('RNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_rnn.history['mae'], label='Training MAE')
plt.plot(history_rnn.history['val_mae'], label='Validation MAE')
plt.title('RNN Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.show()

#### 4.5. Evaluate the RNN Model

In [ ]:
# Evaluate the model on the test set
loss_rnn, mae_rnn = model_rnn.evaluate(X_test_rnn, y_test_rnn, verbose=0)
print(f'Test Loss (MSE): {loss_rnn:.4f}')
print(f'Test MAE: {mae_rnn:.4f}')

# Make predictions
y_pred_rnn = model_rnn.predict(X_test_rnn)

# Visualize predictions vs actual values for a few samples
plt.figure(figsize=(10, 7))
plt.plot(y_test_rnn[:50], label='Actual Values')
plt.plot(y_pred_rnn[:50], label='Predicted Values', linestyle='--')
plt.title('RNN Model: Actual vs Predicted Values (First 50 samples)')
plt.xlabel('Sample Index')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### 5. Convolutional Neural Network (CNN) Example (Image Classification)

We'll use a simple CNN for image classification using a small, synthetic dataset.

#### 5.1. Generate Synthetic Image Data

We'll create a dataset of simple 'shapes' (e.g., squares and circles) for binary classification.

In [ ]:
from sklearn.datasets import make_blobs

def generate_synthetic_images(num_samples, img_size=28, num_classes=2, random_state=42):
    np.random.seed(random_state)
    images = np.zeros((num_samples, img_size, img_size, 1), dtype=np.float32)
    labels = np.zeros(num_samples, dtype=np.int32)

    for i in range(num_samples):
        if np.random.rand() > 0.5: # Class 0: Square
            size = np.random.randint(5, 15)
            x_start = np.random.randint(0, img_size - size)
            y_start = np.random.randint(0, img_size - size)
            images[i, x_start:x_start+size, y_start:y_start+size, 0] = 1.0
            labels[i] = 0
        else: # Class 1: Circle
            radius = np.random.randint(3, 10)
            center_x = np.random.randint(radius, img_size - radius)
            center_y = np.random.randint(radius, img_size - radius)
            Y, X = np.ogrid[-center_x:img_size-center_x, -center_y:img_size-center_y]
            mask = X*X + Y*Y <= radius*radius
            images[i, mask, 0] = 1.0
            labels[i] = 1
    return images, labels

# Generate 500 images of size 28x28
X_cnn, y_cnn = generate_synthetic_images(500, img_size=28)

# Display a few images
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_cnn[i].reshape(28, 28), cmap='gray')
    plt.title(f'Class: {y_cnn[i]}')
    plt.axis('off')
plt.suptitle('Synthetic CNN Image Data (0: Square, 1: Circle)')
plt.show()

#### 5.2. Preprocess Data

Splitting and ensuring correct input shape for CNNs (samples, height, width, channels).

In [ ]:
# Split data into training and testing sets
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(X_cnn, y_cnn, test_size=0.2, random_state=42)

# Data is already scaled (0 or 1) and in the correct shape (samples, height, width, channels)

#### 5.3. Build the CNN Model

We'll use `Conv2D` layers for image processing.

In [ ]:
# Build the model
model_cnn = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model
model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_cnn.summary()

#### 5.4. Train the CNN Model

In [ ]:
# Train the model
history_cnn = model_cnn.fit(X_train_cnn, y_train_cnn, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('CNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Crossentropy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('CNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

#### 5.5. Evaluate the CNN Model

In [ ]:
# Evaluate the model on the test set
loss_cnn, accuracy_cnn = model_cnn.evaluate(X_test_cnn, y_test_cnn, verbose=0)
print(f'Test Loss (Binary Crossentropy): {loss_cnn:.4f}')
print(f'Test Accuracy: {accuracy_cnn:.4f}')

# Make predictions
y_pred_proba_cnn = model_cnn.predict(X_test_cnn)
y_pred_cnn = (y_pred_proba_cnn > 0.5).astype(int)

# Confusion Matrix
cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('CNN Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Classification Report
print('\nCNN Classification Report:')
print(classification_report(y_test_cnn, y_pred_cnn))